In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

PROJECT_PATH = Path.cwd()
if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

import src.model_training as mt
import src.evaluation as ev
from src.data_cleaning import prepare_clean_data
from src.feature_engineering import add_churn_features

RANDOM_STATE = 42
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
FIGURE_PATH = PROJECT_PATH / "reports" / "figures"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)


In [ ]:
raw = pd.read_csv(PROJECT_PATH / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
cleaned = add_churn_features(prepare_clean_data(raw))
cleaned.head()


In [ ]:
X, y, customer_ids = mt.split_features_target(cleaned)
preprocessor = mt.build_preprocessor(X)

log_model = mt.create_logistic_regression_model(preprocessor)
rf_model = mt.create_random_forest_model(preprocessor)
xgb_model = mt.create_XGBoost_model(preprocessor)

X_train, X_test, y_train, y_test, customer_train, customer_test = mt.train_test_split_with_ids(
    X, y, customer_ids
)

log_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

log_metrics, y_pred_log, y_proba_log = ev.evaluate_model(log_model, X_test, y_test, "Logistic Regression")
rf_metrics, y_pred_rf, y_proba_rf = ev.evaluate_model(rf_model, X_test, y_test, "Random Forest")
xgb_metrics, y_pred_xgb, y_proba_xgb = ev.evaluate_model(xgb_model, X_test, y_test, "XGBoost")

results = ev.compare_models([log_metrics, rf_metrics, xgb_metrics])
results.to_csv(PROJECT_PATH / "results" / "model_results.csv", index=False)

cv_scores = cross_val_score(log_model, X_train, y_train, cv=5, scoring="roc_auc")
print("CV ROC-AUC score:", cv_scores)
print("Mean ROC-AUC score:", cv_scores.mean())


In [ ]:
best_model = log_model
y_proba_best = y_proba_log

risk_table = X_test.copy()
risk_table['customer_id'] = customer_test
risk_table['actual_churn'] = y_test.values
risk_table['churn_risk_score'] = y_proba_best
# risk_table.columns
# risk_table.head(10)
risk_table.to_csv(f'{PROCESSED_PATH}/risk_table.csv', index=False)

In [ ]:
threshold_results = []

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_threshold = (y_proba_best >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_test, y_pred_threshold),
        "recall": recall_score(y_test, y_pred_threshold),
        "f1_score": f1_score(y_test, y_pred_threshold)
    })

threshold_results = pd.DataFrame(threshold_results)
threshold_results.to_csv(f'{PROJECT_PATH}/results/threshold_results.csv', index=False)

In [ ]:
# feature importance
best = best_model.named_steps['classifier']
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
# importances = best.feature_importances_
importances = best.coef_[0]

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

feature_importance.head(20)

In [ ]:
top_feature = feature_importance.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_feature['feature'], top_feature['importance'])
plt.xlabel('Features')
plt.ylabel('Importance')
plt.title('Top 15 Positive Logistic Regression Coefficients')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'{FIGURE_PATH}/feature_importance.png', dpi=300)
plt.show()

Month-to-month contracts are strongly associated with churn.
Fiber optic customers appear to have higher churn risk.
Customers without online security or tech support are more likely to churn.
Shorter tenure is associated with higher churn risk.
Electronic check payment method appears among important features.

In [ ]:
# Positive coefficients increase churn probability.
positive_features = feature_importance.sort_values('importance', ascending=False).head(15)
positive_features.to_csv(
    f"{PROJECT_PATH}/results/positive_churn_drivers.csv",
    index=False
)

# Negative coefficients reduce churn probability.
negative_features = feature_importance.sort_values('importance', ascending=True).head(15)
negative_features.to_csv(
    f"{PROJECT_PATH}/results/negative_churn_drivers.csv",
    index=False
)